# Advanced: preference optimization and reliable LLM evaluation

Direct Preference Optimization increases the policy's chosen-versus-rejected log-probability margin relative to a reference model. The second half demonstrates why automated judges need swap tests and uncertainty estimates.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nlp_portfolio.preference import dpo_loss

policy_chosen = np.array([-3.0, -2.2, -4.0, -1.8])
policy_rejected = np.array([-3.4, -2.0, -5.0, -2.1])
reference_chosen = np.array([-3.2, -2.1, -4.4, -1.9])
reference_rejected = np.array([-3.3, -2.2, -4.8, -2.0])

for beta in [0.05, 0.1, 0.5, 1.0]:
    print(beta, np.round(dpo_loss(policy_chosen, policy_rejected,
                                  reference_chosen, reference_rejected, beta), 3))

The reference margin regularizes the update. `beta` controls how strongly preference differences affect the objective; it is not a learning rate. Sequence log probabilities should use the same token masking and length convention for chosen and rejected answers.

In [ ]:
margins = np.linspace(-8, 8, 200)
for beta in [0.1, 0.5, 1.0]:
    loss = dpo_loss(margins, np.zeros_like(margins),
                    np.zeros_like(margins), np.zeros_like(margins), beta)
    plt.plot(margins, loss, label=f"beta={beta}")
plt.xlabel("policy advantage over reference margin")
plt.ylabel("DPO loss")
plt.legend()
plt.show()

## Judge position bias

A pairwise judge can prefer the first response because of order. Every comparison should be repeated with swapped positions. Disagreement marks an unstable judgment rather than a win.

In [ ]:
judgments = pd.DataFrame({
    "example": range(8),
    "A_first": ["A", "A", "B", "A", "B", "A", "B", "A"],
    "B_first": ["B", "A", "B", "B", "B", "A", "A", "B"],
})

def swap_consistent(row):
    # After swapping, a stable preference changes the displayed letter.
    return (row.A_first == "A" and row.B_first == "B") or (row.A_first == "B" and row.B_first == "A")

judgments["order_consistent"] = judgments.apply(swap_consistent, axis=1)
display(judgments)
print("order-consistent fraction:", judgments.order_consistent.mean())

In [ ]:
from nlp_portfolio.evaluation import bootstrap_interval

stable = judgments[judgments.order_consistent]
system_a_wins = (stable.A_first == "A").astype(float).to_numpy()
if len(system_a_wins):
    low, high = bootstrap_interval(system_a_wins, samples=5000, seed=42)
    print("A win rate on stable pairs:", system_a_wins.mean())
    print("bootstrap interval:", (round(low, 3), round(high, 3)))

## Evaluation card

Record the prompt version, decoding parameters, judge model and version, rubric, swap-consistency rate, human agreement on a sampled subset, confidence interval and exclusion rules. Benchmark contamination and evaluator self-preference can otherwise produce a precise-looking score with weak validity.